<a href="https://colab.research.google.com/github/pink3y3/link_prediction-citation_network/blob/main/gae_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Graph AutoEncoder (GAE)

An encoder compresses graph data into a low dimensional representation called an embedding.

A decoder attempts to reconstruct the original graph structure from that representation.

The learned embeddings capture both node features and graph topology.

By reconstructing the graph, GAE can predict missing or future links between nodes.


Graph-Theoretic Heuristic methods don't learn anything. They are called heuristics becasue they're based on human intuition about how networks behave. They have no training, parameters and learning.

GAE and GraphSage are specific types of GNNs.
GNN models learn the rule from the data -> create embeddings -> predict links

Example -
Suppose two papers have CN, but both are about graph neural networks, and have similar abstracts.

Heuristics might say: Probability=low
because they only see the graph not node features

A GNN might say Probability=high because it also uses node features


In [2]:
!pip install torch_geometric
import torch #tensors, neural network utilities, backpropagation
import torch.nn.functional as F #import functional API (activation fns, loss fns and pooling ops)

from torch_geometric.datasets import Planetoid #download and load cora dataset
from torch_geometric.transforms import RandomLinkSplit #split edges into train,val,test
from torch_geometric.nn import GCNConv, GAE
#GCNConv - instead of only looking at itself, node gathers info from its neighbors.
#GAE - wraps encoder and provides functions like model.encode(), model.decode(), model.test() instead of manually writing all of this we write model=GAE(encoder)

from sklearn.metrics import ( #evaluation metrics
    accuracy_score, #percentage of correct predictions tp+tn/total
    precision_score, #out of the links you predicted, how many were actually links tp/tp+fp
    recall_score, # out of all true links, how many did you find? tp/tp+fn
    roc_auc_score, #how well the model separates pos and neg edges VIP
    average_precision_score  #computes area under the precision-recall curve
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.2 MB/s eta 0:00:00


Example: Spam Email Detection

There are 100 emails:
- 20 are actually spam.
- 80 are normal.

Model predictions:
- Predicted 15 emails as spam.
- Out of those 15, 12 were actually spam.

Confusion Matrix:

                          Actually Spam    Actually Not Spam
    Predicted Spam            12                 3
    Predicted Not Spam         8                77



Accuracy = (TP + TN) / Total

         = (12 + 77) / 100

         = 89%
→ Out of 100 emails, 89 were classified correctly.

Precision = TP / (TP + FP)

          = 12 / (12 + 3)

          = 80%
→ Of the 15 emails predicted as spam, 80% were actually spam.

Recall = TP / (TP + FN)

       = 12 / (12 + 8)

       = 60%
→ Of the 20 actual spam emails, the model found 60% of them.

Memory:

Accuracy  = Correct overall.

Precision = Of my YES predictions, how many were right?

Recall    = Of all actual YES cases, how many did I find?

In [3]:
dataset=Planetoid(root='data/Cora', name='Cora')
data=dataset[0]
print(data)

#2708 papers, 1433 features, 10556 directed citation edges

Processing...


Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


Done!


In [4]:
transform = RandomLinkSplit(
    num_val=0.1, #remove edges for val
    num_test=0.1, # remove edges for testing
    is_undirected=True,
    add_negative_train_samples=True # generate fake edges
)

train_data, val_data, test_data = transform(data)


Originally, Cora has directed edges. But, here we often convert it into undirected.
ex: A->B (paper A cites paper B)

A receives information from B only, B doesn't receive info from A.

Undirected, both papers can exchange info during message passing

train_data contains

message passing graph
train_data.edge_index - used by GCN to learn embeddings.

edges to classify
train_data.edge_label_index - contains positive and negative edge pairs.

ground truth

train_data.edge_label - 1 (edge exists), 0 (edge doesn't exist)

In [7]:
#encoder - takes node features and graph structure to give node embeddings(z)

class GCNEncoder(torch.nn.Module):
    def __init__(self,in_channels,hidden_channels,out_channels): #dimensions of network
      super().__init__()

      self.conv1=GCNConv(in_channels,hidden_channels)
      self.conv2=GCNConv(hidden_channels,out_channels)

    def forward(self,x,edge_index):
        x=self.conv1(x,edge_index)
        x=x.relu()
        x=self.conv2(x,edge_index)
        return x


Dimensions of network:

self is the object being created (encoder)
in_channels = 1433;
hidden_channels=64;
out_channels=64;


In [8]:
#create GAE model

encoder=GCNEncoder(dataset.num_features,128,64)
model=GAE(encoder)
print(model)

GAE(
  (encoder): GCNEncoder(
    (conv1): GCNConv(1433, 128)
    (conv2): GCNConv(128, 64)
  )
  (decoder): InnerProductDecoder()
)


In [9]:
optimizer= torch.optim.Adam(model.parameters(),lr=0.01)

In [10]:
def train():
  model.train()
  optimizer.zero_grad()
  z=model.encode(train_data.x,train_data.edge_index)

  loss=model.recon_loss(z,train_data.edge_label_index)
  loss.backward()

  optimizer.step()

  return loss.item()

In [12]:
#train dataset

for epoch in range(200):
  loss=train()

  if epoch%20 == 0:
    print(f"Epoch: {epoch}, Loss: {loss:.4f}")

Epoch: 0, Loss: 1.3595
Epoch: 20, Loss: 1.3215
Epoch: 40, Loss: 1.1833
Epoch: 60, Loss: 1.1123
Epoch: 80, Loss: 1.0592
Epoch: 100, Loss: 1.0313
Epoch: 120, Loss: 1.0090
Epoch: 140, Loss: 0.9875
Epoch: 160, Loss: 0.9665
Epoch: 180, Loss: 0.9512


In [13]:
#testing data
model.eval()

with torch.no_grad():
  z=model.encode(test_data.x,test_data.edge_index)

In [14]:
pred=model.decoder(z, test_data.edge_label_index).sigmoid()

In [15]:
pred_label = (
    pred > 0.5
).int()

In [16]:
y_true = (
    test_data.edge_label
    .cpu()
    .numpy()
)

y_prob = (
    pred
    .cpu()
    .numpy()
)

y_pred = (
    pred_label
    .cpu()
    .numpy()
)

In [17]:
accuracy = accuracy_score(
    y_true,
    y_pred
)

In [18]:
precision = precision_score(
    y_true,
    y_pred
)

In [19]:
recall = recall_score(
    y_true,
    y_pred
)

In [20]:
auc = roc_auc_score(
    y_true,
    y_prob
)

In [21]:
ap = average_precision_score(
    y_true,
    y_prob
)

In [22]:
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print(f"AP       : {ap:.4f}")

Accuracy : 0.5000
Precision: 0.5000
Recall   : 1.0000
ROC-AUC  : 0.8496
AP       : 0.8803
